In [ ]:
import math
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Set

1. Dados Geográficos e Utilitários

In [ ]:
COORDENADAS_ESTADOS = {
    'AC': (-9.9747, -67.8076),
    'AL': (-9.6659, -35.7350),
    'AP': (0.0349, -51.0694),
    'AM': (-3.1186, -60.0212),
    'BA': (-12.9718, -38.5011),
    'CE': (-3.7166, -38.5423),
    'DF': (-15.7795, -47.9297),
    'ES': (-20.3155, -40.3128),
    'GO': (-16.6864, -49.2643),
    'MA': (-2.5387, -44.2825),
    'MT': (-15.6010, -56.0974),
    'MS': (-20.4486, -54.6295),
    'MG': (-19.9208, -43.9378),
    'PA': (-1.4554, -48.4907),
    'PB': (-7.1150, -34.8641),
    'PR': (-25.4284, -49.2733),
    'PE': (-8.0540, -34.8813),
    'PI': (-5.0919, -42.8034),
    'RJ': (-22.9068, -43.1729),
    'RN': (-5.7935, -35.1996),
    'RS': (-30.0346, -51.2177),
    'RO': (-8.7619, -63.9039),
    'RR': (2.8238, -60.6753),
    'SC': (-27.5954, -48.5480),
    'SP': (-23.5505, -46.6333),
    'SE': (-10.9167, -37.0500),
    'TO': (-10.2128, -48.3603)
}

In [ ]:
"""Calcula a distância em km entre dois pontos na Terra."""
def calcular_distancia_haversine(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 6371.0

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2)**2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c




2. Estruturas do Domínio


In [ ]:
@dataclass(frozen=True)
class Pedido:
    id: str
    destino: str
    volume: int
    janela_tempo_max: float # Em horas


In [ ]:
@dataclass
class Veiculo:
    modelo: str
    capacidade_maxima: int  
    velocidade_media: float
    local_atual: str        

In [ ]:
CADASTRO_VEICULOS = {
    "VUC_01": Veiculo("VUC", 50, 70.0, "SP"),
    "TOCO_01": Veiculo("Toco", 100, 80.0, "SP"),
    "TRUCK_01": Veiculo("Truck", 200, 80.0, "MG")
}

In [ ]:
class GrafoTransito:
    def __init__(self):
        self.conexoes: Dict[str, Dict[str, Tuple[float, float]]] = {no: {} for no in COORDENADAS_ESTADOS}

    def adicionar_rodovia(self, origem: str, destino: str, fator_transito: float = 1.0):
        distancia_km = calcular_distancia_haversine(COORDENADAS_ESTADOS[origem], COORDENADAS_ESTADOS[destino])
        self.conexoes[origem][destino] = (distancia_km, fator_transito)
        self.conexoes[destino][origem] = (distancia_km, fator_transito)

    def calcular_tempo_real(self, origem: str, destino: str, velocidade_veiculo: float) -> float:
        distancia, fator = self.conexoes[origem][destino]
        tempo_base = distancia / velocidade_veiculo
        return tempo_base * fator

    def atualizar_transito(self, origem: str, destino: str, novo_fator: float):
        tempo_base = self.conexoes[origem][destino][0]
        self.conexoes[origem][destino] = (tempo_base, novo_fator)
        self.conexoes[destino][origem] = (tempo_base, novo_fator)


3. Modelagem do Estado e A*


In [ ]:
@dataclass(order=True)
class EstadoBusca:
    custo_f: float
    custo_g: float
    local_atual: str = field(compare=False)
    capacidade_restante: int = field(compare=False)
    pedidos_pendentes: frozenset = field(compare=False)
    caminho: List[str] = field(compare=False)

    def is_objetivo(self):
        return len(self.pedidos_pendentes) == 0

def heuristica_geografica(estado: EstadoBusca) -> float:
    """
    Heurística: Tempo estimado em linha reta para a entrega pendente mais distante.
    Velocidade otimista assumida: 100 km/h (para garantir admissibilidade).
    """
    if not estado.pedidos_pendentes:
        return 0.0

    coord_atual = COORDENADAS_ESTADOS[estado.local_atual]
    max_tempo_estimado = 0.0

    for pedido in estado.pedidos_pendentes:
        coord_destino = COORDENADAS_ESTADOS[pedido.destino]
        distancia = calcular_distancia_haversine(coord_atual, coord_destino)
        tempo_estimado = distancia / 100.0
        if tempo_estimado > max_tempo_estimado:
            max_tempo_estimado = tempo_estimado

    return max_tempo_estimado

# def planejar_rota_a_star(origem: str, capacidade: int, pedidos: List[Pedido], mapa: GrafoTransito):
def planejar_rota_a_star(origem: str, veiculo: Veiculo, pedidos: List[Pedido], mapa: GrafoTransito):
    estado_inicial = EstadoBusca(
        custo_f=0, custo_g=0, local_atual=origem,
        capacidade_restante=veiculo.capacidade_maxima, pedidos_pendentes=frozenset(pedidos), caminho=[origem]
    )

    fronteira = []
    heapq.heappush(fronteira, estado_inicial)
    visitados = {}

    while fronteira:
        estado_atual = heapq.heappop(fronteira)

        if estado_atual.is_objetivo():
            return estado_atual.caminho, estado_atual.custo_g

        chave = (estado_atual.local_atual, estado_atual.pedidos_pendentes)
        if chave in visitados and visitados[chave] <= estado_atual.custo_g:
            continue
        visitados[chave] = estado_atual.custo_g

        for vizinho in mapa.conexoes[estado_atual.local_atual]:
            if vizinho not in mapa.conexoes[estado_atual.local_atual]: continue

            tempo_viagem = mapa.calcular_tempo_real(estado_atual.local_atual, vizinho, veiculo.velocidade_media)
            novo_tempo = estado_atual.custo_g + tempo_viagem
            novos_pedidos = set(estado_atual.pedidos_pendentes)
            nova_capacidade = estado_atual.capacidade_restante

            pedido_entregue = next((p for p in novos_pedidos if p.destino == vizinho), None)

            if pedido_entregue:
                if novo_tempo > pedido_entregue.janela_tempo_max: continue # Estourou prazo
                if pedido_entregue.volume > nova_capacidade: continue # Faltou espaço
                novos_pedidos.remove(pedido_entregue)

            novo_estado = EstadoBusca(
                custo_f=0, custo_g=novo_tempo, local_atual=vizinho,
                capacidade_restante=nova_capacidade, pedidos_pendentes=frozenset(novos_pedidos),
                caminho=estado_atual.caminho + [vizinho]
            )
            novo_estado.custo_f = novo_estado.custo_g + heuristica_geografica(novo_estado)
            heapq.heappush(fronteira, novo_estado)

    return None, float('inf')


4. Visualização com Coordenadas Reais


In [ ]:
import folium

def visualizar_mapa_interativo(mapa_transito, rota_destaque=None, trajeto_feito=None):
    # Centraliza o mapa no Brasil (coordenadas médias aproximadas)
    mapa_web = folium.Map(location=[-15.78, -47.93], zoom_start=4, tiles='cartodbpositron')

    # 1. Adicionar todos os estados como pontos de referência
    for estado, coord in COORDENADAS_ESTADOS.items():
        folium.CircleMarker(
            location=coord,
            radius=4,
            color="gray",
            fill=True,
            opacity=0.3,
            popup=estado
        ).add_to(mapa_web)

    # 2. Desenhar o Trajeto já percorrido (Rastro em cinza)
    if trajeto_feito and len(trajeto_feito) > 1:
        pontos_rastro = [COORDENADAS_ESTADOS[loc] for loc in trajeto_feito]
        folium.PolyLine(
            pontos_rastro,
            color="#7F8C8D",
            weight=3,
            opacity=0.6,
            dash_array='5, 10'
        ).add_to(mapa_web)

    # 3. Destacar a Rota Planejada pelo A* (Linha vermelha para o futuro)
    if rota_destaque and len(rota_destaque) > 1:
        pontos_rota = [COORDENADAS_ESTADOS[loc] for loc in rota_destaque]
        folium.PolyLine(
            pontos_rota,
            color="#E74C3C",
            weight=5,
            opacity=0.8
        ).add_to(mapa_web)

    # 4. Ícone do Caminhão na posição atual
    # A posição atual é o primeiro ponto da rota futura (ou o último do trajeto feito)
    posicao_atual = None
    if rota_destaque:
        posicao_atual = COORDENADAS_ESTADOS[rota_destaque[0]]
    elif trajeto_feito:
        posicao_atual = COORDENADAS_ESTADOS[trajeto_feito[-1]]

    if posicao_atual:
        folium.Marker(
            location=posicao_atual,
            icon=folium.Icon(color='red', icon='truck', prefix='fa'),
            popup="Caminhão"
        ).add_to(mapa_web)

    return mapa_web


5. Simulador Interativo com Gráficos


In [ ]:
from IPython.display import clear_output, display

def simulador_interativo_com_grafos(mapa: GrafoTransito, origem_inicial: str, veiculo: Veiculo, pedidos_iniciais: List[Pedido]):
    local_atual = origem_inicial
    pedidos_pendentes = list(pedidos_iniciais)
    tempo_acumulado = 0.0
    capacidade_atual = veiculo.capacidade_maxima
    trajeto_percorrido = [origem_inicial]

    while pedidos_pendentes:
        # Limpa a tela para atualizar o dashboard e o mapa
        clear_output(wait=True)

        # 1. Planejamento com o A*
        rota, _ = planejar_rota_a_star(
            origem=local_atual,
            veiculo=veiculo,
            pedidos=pedidos_pendentes,
            mapa=mapa
        )

        if not rota:
            print("🚨 ALERTA: Rota válida não encontrada!")
            break

        # 2. Painel de Status (Dashboard)
        print("="*55)
        print(f"🚚 VEÍCULO EM: {local_atual} | ⏱️ TEMPO: {tempo_acumulado:.2f}h")
        print(f"📦 Carga: {veiculo.capacidade_maxima - capacidade_atual}/{veiculo.capacidade_maxima}")
        print(f"🛣️  TRAJETO FEITO: {' -> '.join(trajeto_percorrido)}")
        print(f"🎯 ROTA PLANEJADA: {' -> '.join(rota)}")
        print("="*55)

        # 3. RENDERIZA O MAPA INTERATIVO NO LUGAR DO MATPLOTLIB
        mapa_folium = visualizar_mapa_interativo(
            mapa, 
            rota_destaque=rota, 
            trajeto_feito=trajeto_percorrido
        )
        display(mapa_folium)

        # 4. Interação do Usuário
        print("\n⏳ Aguardando comando...")
        acao = input("Pressione [ENTER] para seguir viagem ou [S] para reportar trânsito: ").strip().lower()

        if acao == 's':
            # Código de atualização de trânsito existente no notebook algoritmo_2.ipynb
            print(f"Rodovias conectadas a {local_atual}: {list(mapa.conexoes[local_atual].keys())}")
            rod_orig = input("Origem (Ex: SP): ").strip().upper()
            rod_dest = input("Destino (Ex: MG): ").strip().upper()
            try:
                fator = float(input("Novo fator de lentidão (Ex: 3.0): "))
                mapa.atualizar_transito(rod_orig, rod_dest, fator)
                continue # Recalcula a rota sem mover o caminhão
            except:
                input("Entrada inválida. [ENTER] para continuar...")
                continue

        # 5. Movimentação Física
        proximo_destino = rota[1]
        tempo_viagem = mapa.calcular_tempo_real(local_atual, proximo_destino, veiculo.velocidade_media)
        
        local_atual = proximo_destino
        tempo_acumulado += tempo_viagem
        trajeto_percorrido.append(local_atual)

        # 6. Verificação de Entregas
        entregues = [p for p in pedidos_pendentes if p.destino == local_atual]
        for p in entregues:
            if tempo_acumulado <= p.janela_tempo_max:
                print(f"✅ SUCESSO: Pedido {p.id} entregue no prazo!")
            else:
                print(f"❌ ATRASO: Pedido {p.id} entregue fora do prazo!")
            capacidade_atual += p.volume
            pedidos_pendentes.remove(p)

        if pedidos_pendentes:
            input("\nPressione [ENTER] para continuar...")

    # Finalização
    clear_output(wait=True)
    print("🏁 TODAS AS ENTREGAS CONCLUÍDAS")
    mapa_final = visualizar_mapa_interativo(mapa, trajeto_feito=trajeto_percorrido)
    display(mapa_final)


6. Executando



In [ ]:
# Inicialização do Grafo
mapa_br = GrafoTransito()

# ==========================================
# 1. REGIÃO SUL
# ==========================================
mapa_br.adicionar_rodovia('RS', 'SC')
mapa_br.adicionar_rodovia('SC', 'PR')

# ==========================================
# 2. REGIÃO SUDESTE
# ==========================================
mapa_br.adicionar_rodovia('PR', 'SP') # Conexão Sul-Sudeste
mapa_br.adicionar_rodovia('SP', 'RJ')
mapa_br.adicionar_rodovia('SP', 'MG')
mapa_br.adicionar_rodovia('SP', 'MS') # Conexão Sudeste-Centro-Oeste
mapa_br.adicionar_rodovia('RJ', 'MG')
mapa_br.adicionar_rodovia('RJ', 'ES')
mapa_br.adicionar_rodovia('MG', 'ES')
mapa_br.adicionar_rodovia('MG', 'GO') # Conexão Sudeste-Centro-Oeste
mapa_br.adicionar_rodovia('MG', 'BA') # Conexão Sudeste-Nordeste
mapa_br.adicionar_rodovia('ES', 'BA') # Conexão Sudeste-Nordeste

# ==========================================
# 3. REGIÃO CENTRO-OESTE
# ==========================================
mapa_br.adicionar_rodovia('MS', 'MT')
mapa_br.adicionar_rodovia('MS', 'GO')
mapa_br.adicionar_rodovia('MT', 'GO')
mapa_br.adicionar_rodovia('MT', 'RO') # Conexão Centro-Oeste-Norte
mapa_br.adicionar_rodovia('MT', 'PA') # Conexão Centro-Oeste-Norte
mapa_br.adicionar_rodovia('MT', 'AM') # Conexão Centro-Oeste-Norte
mapa_br.adicionar_rodovia('GO', 'DF')
mapa_br.adicionar_rodovia('GO', 'TO') # Conexão Centro-Oeste-Norte
mapa_br.adicionar_rodovia('GO', 'BA') # Conexão Centro-Oeste-Nordeste

# ==========================================
# 4. REGIÃO NORDESTE
# ==========================================
mapa_br.adicionar_rodovia('BA', 'SE')
mapa_br.adicionar_rodovia('BA', 'AL')
mapa_br.adicionar_rodovia('BA', 'PE')
mapa_br.adicionar_rodovia('BA', 'PI')
mapa_br.adicionar_rodovia('SE', 'AL')
mapa_br.adicionar_rodovia('AL', 'PE')
mapa_br.adicionar_rodovia('PE', 'PB')
mapa_br.adicionar_rodovia('PE', 'CE')
mapa_br.adicionar_rodovia('PE', 'PI')
mapa_br.adicionar_rodovia('PB', 'RN')
mapa_br.adicionar_rodovia('PB', 'CE')
mapa_br.adicionar_rodovia('RN', 'CE')
mapa_br.adicionar_rodovia('CE', 'PI')
mapa_br.adicionar_rodovia('PI', 'MA')
mapa_br.adicionar_rodovia('PI', 'TO') # Conexão Nordeste-Norte
mapa_br.adicionar_rodovia('MA', 'PA') # Conexão Nordeste-Norte
mapa_br.adicionar_rodovia('MA', 'TO') # Conexão Nordeste-Norte

# ==========================================
# 5. REGIÃO NORTE
# ==========================================
mapa_br.adicionar_rodovia('TO', 'PA')
mapa_br.adicionar_rodovia('PA', 'AP')
mapa_br.adicionar_rodovia('PA', 'RR')
mapa_br.adicionar_rodovia('PA', 'AM')
mapa_br.adicionar_rodovia('AM', 'RR')
mapa_br.adicionar_rodovia('AM', 'AC')
mapa_br.adicionar_rodovia('AM', 'RO')
mapa_br.adicionar_rodovia('RO', 'AC')

In [ ]:

pedidos_hoje = [
    Pedido("P1", "ES", volume=20, janela_tempo_max=24.0),
    Pedido("P2", "SC", volume=30, janela_tempo_max=48.0)
]

In [ ]:
veiculo_escolhido = CADASTRO_VEICULOS["TOCO_01"]

In [ ]:
simulador_interativo_com_grafos(mapa_br, origem_inicial='SP', veiculo=veiculo_escolhido, pedidos_iniciais=pedidos_hoje)